# Lesson 5: Working with Real Data

**Duration:** 60 minutes

## Learning Objectives
- Handle messy, real-world datasets
- Address class imbalance issues
- Debug common data and training problems
- Implement data augmentation strategies
- Evaluate model robustness

## 5.1 Common Data Issues

**Real-world datasets often have:**
- Missing or corrupted images
- Inconsistent image sizes
- Class imbalance (more of one class than another)
- Mislabeled samples
- Low resolution or noise

**Strategies to handle:**
1. Data cleaning and validation
2. Stratified sampling for balanced splits
3. Weighted loss functions for imbalanced classes
4. Data augmentation to increase diversity

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt

# Load CIFAR-10 to simulate class imbalance
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# Check class distribution
labels = np.array(train_dataset.targets)
unique, counts = np.unique(labels, return_counts=True)
class_dist = dict(zip(unique, counts))
print('Original class distribution:')
for cls, count in class_dist.items():
    print(f'  Class {cls}: {count} samples')

# Visualize
plt.figure(figsize=(10, 4))
plt.bar(class_dist.keys(), class_dist.values())
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution in CIFAR-10')
plt.grid(True)
plt.show()

## 5.2 Handling Class Imbalance

**Weighted Sampling:** Over-sample under-represented classes

**Weighted Loss:** Penalize misclassification of rare classes

In [ ]:
["# Method 1: Weighted Random Sampler\nclass_weights = 1. / torch.tensor(np.bincount(labels).astype('float'))\nsample_weights = class_weights[labels]\nsampler = WeightedRandomSampler(sample_weights, len(sample_weights))\n\ntrain_loader_balanced = DataLoader(train_dataset, batch_size=32, sampler=sampler)\n\n# Check balanced batches\nimages, batch_labels = next(iter(train_loader_balanced))\nprint(f'Balanced batch distribution: {torch.bincount(torch.tensor(batch_labels)).numpy()}')\n\n# Method 2: Weighted Loss\nclass_weights_loss = 1. / np.array([class_dist[i] for i in range(10)])\nclass_weights_loss = class_weights_loss / class_weights_loss.sum() * 10\nweights = torch.tensor(class_weights_loss, dtype=torch.float)\nweighted_loss = nn.CrossEntropyLoss(weight=weights)\n\nprint(f'\\nClass weights for loss: {weights}')\nprint('Rare classes will be weighted more heavily in the loss')"]]

## 5.3 Data Augmentation

Augmentation increases dataset diversity and robustness:

In [ ]:
["# Define augmentation transforms\naugment_transform = transforms.Compose([\n    transforms.RandomRotation(15),\n    transforms.RandomHorizontalFlip(p=0.5),\n    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),\n    transforms.ColorJitter(brightness=0.2, contrast=0.2),\n    transforms.RandomCrop(32, padding=4),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])\n])\n\naugmented_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=augment_transform)\n\n# Visualize augmented samples\nfig, axes = plt.subplots(3, 4, figsize=(12, 9))\nfor i in range(3):\n    img, _ = augmented_dataset[0]\n    for j in range(4):\n        if j == 0:\n            axes[i, j].imshow((img.transpose(0, 2).transpose(0, 1).numpy() * 0.5 + 0.5))\n        else:\n            img_aug, _ = augmented_dataset[0]\n            axes[i, j].imshow((img_aug.transpose(0, 2).transpose(0, 1).numpy() * 0.5 + 0.5))\n        axes[i, j].axis('off')\n\nplt.suptitle('Data Augmentation Examples')\nplt.tight_layout()\nplt.show()\n\nprint('Augmentation applied: Rotation, Flips, Crops, Color Jitter')\nprint('This helps the model generalize better to new data')"]]